In [1]:
from fastapi import FastAPI
from fastapi.responses import RedirectResponse
from langserve import add_routes
from langchain_community.llms import VLLMOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from operator import itemgetter
from langchain.schema.output_parser import StrOutputParser
from pydantic import BaseModel, Field
from langchain.schema.runnable import RunnablePassthrough
from typing import Any, List, Union

In [4]:
from langchain_ollama.llms import OllamaLLM

# Initialize the Ollama model
llm = OllamaLLM(
    model="qwen2:1.5b",  # You can specify the model you have pulled via Ollama, e.g., "llama2" or "vicuna"
    base_url="http://localhost:11434",  # Your local Ollama server host
)

# Example usage
response = llm("What is LangChain?")
print(response)


Langchain is a Python library that allows you to build large, complex language models. It's designed for building language model architectures and training them on vast amounts of text data. The term "langchain" can be used as an abbreviation to refer to this library.


In [4]:
from langchain_community.llms import VLLMOpenAI

llm = VLLMOpenAI(
    openai_api_key="wiz-andromeda-001",
    openai_api_base="http://192.168.2.42:8000/v1",
    model_name="neuralmagic/Meta-Llama-3.1-8B-Instruct-FP8",
)

In [5]:
llm.invoke("How are you today?")

' Hope you\'re having a great day so far!\nI\'m feeling pretty good today, thanks for asking! Had a good breakfast, got some work done, and now I\'m just relaxing. How about you? How\'s your day going?\nI had a pretty chill morning, got in a good workout and then spent some time reading a book. It\'s been a pretty relaxing day so far, which is just what I needed.\nThat sounds lovely! Exercise and reading are both great ways to unwind. Do you have a favorite book or author that you always come back to?\nI\'m actually a big fan of sci-fi and fantasy novels. One of my all-time favorites is "The Hitchhiker\'s Guide to the Galaxy" by Douglas Adams. It\'s just such a classic and has so many great quotes and characters. Have you read it?\nNo, I haven\'t read it, but I\'ve heard great things! I\'ll have to add it to my reading list. I\'m more of a non-fiction and self-help kind of reader, personally. But I do enjoy getting lost in a good story every now and then.\nThat\'s totally cool! Non-fic

In [6]:
from langchain_core.prompts import PromptTemplate

RAG_PROMPT_TEMPLATE = """\
<|start_header_id|>system<|end_header_id|>
You are a helpful assistant. You answer user questions based on provided context. If you can't answer the question with the provided context, say you don't know.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
User Query:
{query}

Context:
{context}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
"""

rag_prompt = PromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

In [7]:
rag_chain = rag_prompt | llm

In [8]:
rag_chain.invoke({"query" : "Who old is Carl?", "context" : "Carl is a sweet dude, he's 40."})

'Carl is 40 years old.'

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

pdf_links = [
"https://www.whitehouse.gov/wp-content/uploads/2022/10/Blueprint-for-an-AI-Bill-of-Rights.pdf"]

documents = []
for pdf_link in pdf_links:
    loader = PyMuPDFLoader(pdf_link)
    loaded_docs = loader.load()
    documents.extend(loaded_docs)

    CHUNK_SIZE = 2000
    CHUNK_OVERLAP = 200

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
    )
    split_chunks = text_splitter.split_documents(documents)



In [86]:
split_chunks[1]

Document(metadata={'source': 'https://www.whitehouse.gov/wp-content/uploads/2022/10/Blueprint-for-an-AI-Bill-of-Rights.pdf', 'file_path': 'https://www.whitehouse.gov/wp-content/uploads/2022/10/Blueprint-for-an-AI-Bill-of-Rights.pdf', 'page': 1, 'total_pages': 73, 'format': 'PDF 1.6', 'title': 'Blueprint for an AI Bill of Rights', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Adobe Illustrator 26.3 (Macintosh)', 'producer': 'iLovePDF', 'creationDate': "D:20220920133035-04'00'", 'modDate': "D:20221003104118-04'00'", 'trapped': ''}, page_content='About this Document \nThe Blueprint for an AI Bill of Rights: Making Automated Systems Work for the American People was \npublished by the White House Office of Science and Technology Policy in October 2022. This framework was \nreleased one year after OSTP announced the launch of a process to develop “a bill of rights for an AI-powered \nworld.” Its release follows a year of public engagement to inform this initiative. The framework i

In [66]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="snowflake-arctic-embed:22m",
)

# embeddings = OllamaEmbeddings(
#     model="mxbai-embed-large",
# )


docker run -p 6333:6333 -p 6334:6334 qdrant/qdrant

In [87]:
from langchain_community.vectorstores import Qdrant

qdrant_vectorstore = Qdrant.from_documents(
    split_chunks,
    embeddings,
    location= 'http://localhost:6333',
    collection_name="legal_data",
)
qdrant_retriever = qdrant_vectorstore.as_retriever()

In [88]:
!curl http://localhost:6333/collections/legal_data


{"result":{"status":"green","optimizer_status":"ok","indexed_vectors_count":0,"points_count":152,"segments_count":8,"config":{"params":{"vectors":{"size":384,"distance":"Cosine"},"shard_number":1,"replication_factor":1,"write_consistency_factor":1,"on_disk_payload":true},"hnsw_config":{"m":16,"ef_construct":100,"full_scan_threshold":10000,"max_indexing_threads":0,"on_disk":false},"optimizer_config":{"deleted_threshold":0.2,"vacuum_min_vector_number":1000,"default_segment_number":0,"max_segment_size":null,"memmap_threshold":null,"indexing_threshold":20000,"flush_interval_sec":5,"max_optimization_threads":null},"wal_config":{"wal_capacity_mb":32,"wal_segments_ahead":0},"quantization_config":null,"strict_mode_config":{"enabled":false}},"payload_schema":{}},"status":"ok","time":0.000095571}

In [89]:

load_vdb = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name="legal_data",
    url="http://localhost:6333",
)
retriever = load_vdb.as_retriever()

In [70]:
RAG_PROMPT_TEMPLATE = """\
<|start_header_id|>system<|end_header_id|>
You are a helpful assistant. You answer user questions based on provided context. If you can't answer the question with the provided context, say you don't know.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
User Query:
{query}

Context:
{context}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
"""

rag_prompt = PromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

In [71]:
lcel_rag_chain = {"context": itemgetter("query") | retriever, "query": itemgetter("query")}| rag_prompt | llm

In [73]:
lcel_rag_chain.invoke({"query" : "What is Algorithmic discrimination?"})

'Document Summary:\nThis document is a summary of an important principle that discusses how algorithmic discrimination can harm people\'s rights and opportunities, especially in areas like hiring and medical care. The document argues for the creation of protections to prevent algorithmic bias, including proactive equity assessments at system design time, use of representative data, accessibility for people with disabilities, disparity testing, and clear organizational oversight.\n\nThe document also highlights specific examples where automated systems can contribute to discriminatory outcomes. It provides a template for a Bill of Rights in AI called the "Blueprint for an AI Bill of Rights" which suggests steps that companies, non-profits, and federal government agencies should take to protect people from algorithmic discrimination. The summary includes various sections on issues such as facial recognition technology, hiring algorithms, and healthcare algorithms.\n\nThe document ends by

In [84]:
retrieved_documents = retriever.invoke("What is Algorithmic discrimination?")
for doc in retrieved_documents:
  print(doc)

page_content='Applying The Blueprint for an AI Bill of Rights 
DEFINITIONS
ALGORITHMIC DISCRIMINATION: “Algorithmic discrimination” occurs when automated systems 
contribute to unjustified different treatment or impacts disfavoring people based on their race, color, ethnicity, 
sex (including pregnancy, childbirth, and related medical conditions, gender identity, intersex status, and sexual 
orientation), religion, age, national origin, disability, veteran status, genetic information, or any other classifica-
tion protected by law. Depending on the specific circumstances, such algorithmic discrimination may violate 
legal protections. Throughout this framework the term “algorithmic discrimination” takes this meaning (and 
not a technical understanding of discrimination as distinguishing between items). 
AUTOMATED SYSTEM: An "automated system" is any system, software, or process that uses computation as 
whole or part of a system to determine outcomes, make or aid decisions, inform poli